In [2]:
!git clone https://github.com/eliftufekci/k_shortest_path_with_diversity.git

fatal: destination path 'k_shortest_path_with_diversity' already exists and is not an empty directory.


In [3]:
import sys

top_level_container_dir = "k_shortest_path_with_diversity/"

if top_level_container_dir not in sys.path:
    sys.path.insert(0, top_level_container_dir)

In [ ]:
import gc
import networkx as nx
import random
import datetime
import numpy as np

from k_shortest_path_with_diversity.examples import draw_line_chart, download_and_prepare_graphs
from k_shortest_path_with_diversity.examples.draw_distribution import draw_time_distribution, draw_num_of_path_distribution
from k_shortest_path_with_diversity.src.algorithms import FindKSPD, FindKSPD_Minus

In [ ]:
def run_algorithm(algorithm, G, threshold, k, node_pairs):
    times = []
    num_paths = []

    for src, dest in node_pairs:
        print(f"\nComparing algorithms for SRC: {src}, DEST: {dest}")

        start_time = datetime.datetime.now()
        alg = algorithm(G, threshold)
        result = alg.find_paths(src=src, dest=dest, k=k)
        end_time = datetime.datetime.now()
        execution_time = end_time - start_time

        times.append(execution_time.total_seconds())
        num_paths.append(alg.number_of_paths_explored)

    return times, num_paths

In [6]:
def upload_graph(filepath, num_pairs):
    G = nx.DiGraph()
    with open(filepath) as f:
        for line in f:
            parts = line.split()
            if len(parts) == 3:  # Weighted graph (3 parçalı satır: kaynak, hedef, ağırlık)
                u, v, weight = int(parts[0]), int(parts[1]), float(parts[2])
                G.add_edge(u, v, weight=weight)
            elif len(parts) == 2:  # Unweighted graph (2 parçalı satır: kaynak ve hedef)
                u, v = map(int, parts)
                G.add_edge(u, v, weight=1)  # Ağırlıksız kenarları varsayılan olarak '1' ağırlıkla ekle
            else:
                raise ValueError("Unexpected line format in graph file.")

    node_pairs = []
    for _ in range(num_pairs):
        src = random.choice(list(G.nodes()))
        reachable = list(nx.descendants(G, src))

        while not reachable:
            src = random.choice(list(G.nodes()))
            reachable = list(nx.descendants(G, src))

        dest = random.choice(reachable)
        node_pairs.append((src, dest))

    return G, node_pairs

In [ ]:
download_and_prepare_graphs()

k_list = [5, 10, 15, 20]
diversity_threshold = 0.6 #not important

Downloading: https://snap.stanford.edu/data/web-Google.txt.gz
Extracting: /content/graph-data/web-Google.txt.gz -> /content/graph-data/web-Google.txt
Removed archive: /content/graph-data/web-Google.txt.gz
Done: /content/graph-data/web-Google.txt
Downloading: https://snap.stanford.edu/data/wiki-Talk.txt.gz
Extracting: /content/graph-data/wiki-Talk.txt.gz -> /content/graph-data/wiki-Talk.txt
Removed archive: /content/graph-data/wiki-Talk.txt.gz
Done: /content/graph-data/wiki-Talk.txt
Downloading: https://www.diag.uniroma1.it/challenge9/data/USA-road-d/USA-road-d.FLA.gr.gz
Extracting: /content/graph-data/USA-road-d.FLA.gr.gz -> /content/graph-data/USA-road-d.FLA.gr
Removed archive: /content/graph-data/USA-road-d.FLA.gr.gz
Done: /content/graph-data/USA-road-d.FLA.gr
Downloading: https://www.diag.uniroma1.it/challenge9/data/USA-road-d/USA-road-d.COL.gr.gz
Extracting: /content/graph-data/USA-road-d.COL.gr.gz -> /content/graph-data/USA-road-d.COL.gr
Removed archive: /content/graph-data/USA-ro

In [ ]:
roadFLA_path = "/content/graph-data/USA-road-d.FLA.gr"
num_pairs = 5
G, node_pairs = upload_graph(roadFLA_path, num_pairs)
print(node_pairs)

In [ ]:
all_results = []

for k_to_find in k_list:    
    print("working with KSPD")
    kspd_times, kspd_num_paths = run_algorithm(FindKSPD, G, diversity_threshold, k_to_find, node_pairs)

    print(f"kspd_times: {kspd_times}")
    print(f"kspd_num_paths: {kspd_num_paths}")

    print("working with KSPD_Minus")
    kspd_minus_times, kspd_minus_num_paths = run_algorithm(FindKSPD_Minus, G, diversity_threshold, k_to_find, node_pairs)

    print(f"kspd_minus_times: {kspd_minus_times}")
    print(f"kspd_minus_num_paths: {kspd_minus_num_paths}")

    roadFLA_result = (
            (np.average(kspd_times) if kspd_times else 0,
            np.average(kspd_num_paths) if kspd_num_paths else 0,
            kspd_times, kspd_num_paths),
            (np.average(kspd_minus_times) if kspd_minus_times else 0,
            np.average(kspd_minus_num_paths) if kspd_minus_num_paths else 0,
            kspd_minus_times, kspd_minus_num_paths)
        )
    print(roadFLA_result)

    all_results.append(roadFLA_result)

In [ ]:
graph_types = ("RoadFLA",)

algorithms_paths = {
    'FindKSPD':       [r[0][1] for r in all_results],  # index 1 = avg_num_paths
    'FindKSPD_Minus': [r[1][1] for r in all_results],
}

algorithms_time = {
    'FindKSPD':       [r[0][0] for r in all_results],  # index 0 = avg_time
    'FindKSPD_Minus': [r[1][0] for r in all_results],
}

markers = {
    'FindKSPD':       's',  # □ kare
    'FindKSPD_Minus': '^',  # △ üçgen
}

draw_line_chart(k_list, markers, algorithms_paths, algorithms_time, graph_name="RoadFLA")

In [ ]:
all_kspd_times = []
all_kspd_minus_times = []
all_kspd_num_paths = []
all_kspd_minus_num_paths = []

for result in all_results:
    all_kspd_times.extend(result[0][2])
    all_kspd_num_paths.extend(result[0][3])
    all_kspd_minus_times.extend(result[1][2])
    all_kspd_minus_num_paths.extend(result[1][3])

# Plotting distributions for times
plot_configs_time = [
    ('FindKSPD Execution Times', all_kspd_times, 'skyblue'),
    ('FindKSPD_Minus Execution Times', all_kspd_minus_times, 'lightcoral'),
]
draw_time_distribution(plot_configs_time)

# Plotting distributions for number of paths
plot_configs_num_paths = [
    ('FindKSPD Number of Paths Explored', all_kspd_num_paths, 'skyblue'),
    ('FindKSPD_Minus Number of Paths Explored', all_kspd_minus_num_paths, 'lightcoral'),
]
draw_num_of_path_distribution(plot_configs_num_paths)